# RFP Response Automation
## Parse, Retrieve Prior Answers, Draft Responses, Flag Gaps

**Project -- NLP / Enterprise Workflow Portfolio**

---

Responding to Requests for Proposals (RFPs) is one of the most time-consuming
tasks in enterprise sales. A single RFP can contain 50-200 questions spanning
security, compliance, architecture, pricing, and SLAs. Companies answer the
**same questions** across dozens of RFPs per quarter -- yet each response is
drafted from scratch.

This notebook builds a complete **RFP response pipeline**:

1. **Parse** -- Extract structured questions from raw RFP text
2. **Retrieve** -- Find the best prior answers from a knowledge base
3. **Draft** -- Generate tailored responses using context-aware synthesis
4. **Flag** -- Identify questions with no good prior answer (human review needed)

The design follows enterprise workflow patterns used in production RFP tools.

## Table of Contents

| Part | Topic | What It Does |
|---|---|---|
| 1 | Enterprise workflow design | Architecture and design decisions |
| 2 | RFP parser | Extract questions from raw text |
| 3 | Prior answer knowledge base | 60 Q&A pairs across 8 categories |
| 4 | Retriever | Hybrid BM25 + dense search over prior answers |
| 5 | Response drafter | Context-aware answer synthesis |
| 6 | Gap flagger | Detect unanswered / low-confidence sections |
| 7 | End-to-end pipeline | Full RFP processing |
| 8 | Evaluation | Quality metrics across categories |
| 9 | Visualisations | Pipeline performance analysis |
| 10 | Production patterns | Real-world deployment guidance |

## Part 1: Enterprise Workflow Design

### Why RFP Response Is Hard

| Challenge | Impact |
|---|---|
| Questions are unstructured free text | Parser must handle varied formats |
| Same topic worded differently across RFPs | Retriever needs semantic matching |
| Answers must be tailored per client | Cannot just copy-paste prior answers |
| Some questions are novel | System must flag gaps, not hallucinate |
| Compliance / legal sensitivity | Every response needs human review |

### System Architecture

```
  ┌──────────────────────────────────────────────────────────────────┐
  │                      RFP RESPONSE PIPELINE                      │
  │                                                                  │
  │   ┌──────────┐    ┌───────────┐    ┌──────────┐   ┌──────────┐ │
  │   │  PARSER   │───>│ RETRIEVER │───>│ DRAFTER  │──>│ FLAGGER  │ │
  │   │          │    │           │    │          │   │          │ │
  │   │ Raw RFP  │    │ Prior     │    │ Tailored │   │ Coverage │ │
  │   │ -> Qns   │    │ Answer KB │    │ Drafts   │   │ Report   │ │
  │   └──────────┘    └───────────┘    └──────────┘   └──────────┘ │
  │        │                │                │              │       │
  │        ▼                ▼                ▼              ▼       │
  │   Structured       Ranked            Draft          Flagged     │
  │   questions        matches           responses      gaps        │
  └──────────────────────────────────────────────────────────────────┘
                              │
                              ▼
                     Human Review Queue
                     (flagged items first)
```

### Design Decisions

| Decision | Rationale |
|---|---|
| **Hybrid retrieval** (BM25 + dense) | Technical terms need exact match; paraphrased questions need semantic match |
| **Confidence scoring** | Every draft gets a score -- low confidence triggers human review |
| **Category tagging** | RFP questions are grouped by domain (security, SLA, etc.) |
| **Human-in-the-loop** | System drafts, humans approve -- never auto-submit |
| **Prior answer versioning** | Knowledge base is append-only with timestamps |

## Environment Setup

In [ ]:
import subprocess, sys
for pkg in ['pandas', 'numpy', 'matplotlib', 'seaborn', 'plotly']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
print('Ready.')

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import re, math, json as json_mod, textwrap, hashlib, copy
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from typing import Optional
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

pd.set_option('display.max_columns', 25)
pd.set_option('display.max_colwidth', 140)
plt.rcParams['figure.figsize'] = (14, 5)
sns.set_style('whitegrid')
np.random.seed(42)
print('Imports OK')

## Part 2: RFP Parser

RFP documents arrive as unstructured text with inconsistent formatting.
The parser extracts individual questions, detects their category, and
assigns section numbers.

### Parsing Challenges

| Pattern | Example |
|---|---|
| Numbered questions | `3.1 Describe your backup strategy.` |
| Bullet-point questions | `- What certifications do you hold?` |
| Multi-sentence questions | Long paragraph ending with `?` |
| Implicit questions | `Provide details of your SLA guarantees.` |
| Section headers mixed with questions | `Security: How do you encrypt data at rest?` |

In [ ]:
CATEGORY_KEYWORDS = {
    'security': ['security', 'encrypt', 'authentication', 'authorization',
                 'vulnerability', 'penetration', 'soc2', 'soc 2', 'iso 27001',
                 'gdpr', 'access control', 'firewall', 'intrusion', 'mfa',
                 'audit log', 'zero trust', 'rbac'],
    'compliance': ['compliance', 'regulation', 'certif', 'audit', 'hipaa',
                   'pci', 'gdpr', 'ccpa', 'fedramp', 'data residency',
                   'data sovereignty', 'privacy', 'retention'],
    'architecture': ['architecture', 'infrastructure', 'scalab', 'microservice',
                     'api', 'database', 'cloud', 'aws', 'azure', 'gcp',
                     'kubernetes', 'container', 'serverless', 'caching'],
    'sla': ['sla', 'uptime', 'availability', 'downtime', 'rto', 'rpo',
            'disaster recovery', 'failover', 'redundan', 'backup',
            'maintenance window', 'incident response'],
    'integration': ['integration', 'api', 'webhook', 'sso', 'saml', 'oauth',
                    'rest', 'graphql', 'sdk', 'connector', 'migration',
                    'import', 'export', 'interoperab'],
    'data_handling': ['data', 'storage', 'backup', 'migration', 'retention',
                      'deletion', 'portability', 'format', 'etl', 'pipeline',
                      'warehouse', 'lake'],
    'support': ['support', 'sla', 'response time', 'ticket', 'escalation',
                'account manager', 'onboarding', 'training', 'documentation',
                'customer success'],
    'pricing': ['pricing', 'cost', 'license', 'subscription', 'tier',
                'discount', 'volume', 'contract', 'payment', 'invoice',
                'commitment', 'overage'],
}


@dataclass
class RFPQuestion:
    section_id: str
    text: str
    category: str
    original_line: int


def classify_category(text):
    tl = text.lower()
    scores = {}
    for cat, kws in CATEGORY_KEYWORDS.items():
        scores[cat] = sum(1 for kw in kws if kw in tl)
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else 'general'


def parse_rfp(rfp_text):
    lines = rfp_text.strip().split('\n')
    questions = []
    section_counter = defaultdict(int)

    for i, line in enumerate(lines):
        line = line.strip()
        if not line or len(line) < 10:
            continue

        # Detect question patterns
        is_question = False
        clean = line

        # Pattern: numbered (3.1, 3.1.1, etc.)
        num_match = re.match(r'^(\d+(?:\.\d+)*)[\.\)]?\s+(.+)', line)
        if num_match:
            clean = num_match.group(2)
            is_question = True

        # Pattern: bullet points
        bullet_match = re.match(r'^[-*\u2022]\s+(.+)', line)
        if bullet_match:
            clean = bullet_match.group(1)
            is_question = True

        # Pattern: ends with ? or starts with action verbs
        if clean.endswith('?'):
            is_question = True
        if re.match(r'^(describe|explain|provide|detail|list|outline|specify|what|how|does|do|is|are|can|will)\b', clean, re.I):
            is_question = True

        if is_question:
            category = classify_category(clean)
            section_counter[category] += 1
            section_id = f'{category.upper()}-{section_counter[category]:03d}'
            questions.append(RFPQuestion(
                section_id=section_id,
                text=clean,
                category=category,
                original_line=i + 1,
            ))

    return questions


print(f'Parser ready | {len(CATEGORY_KEYWORDS)} categories')

In [ ]:
# Sample RFP document
SAMPLE_RFP = """
REQUEST FOR PROPOSAL: Enterprise Cloud Platform
Issued by: GlobalCorp Financial Services
Due Date: 2026-05-15

SECTION 1: SECURITY

1.1 Describe your encryption approach for data at rest and data in transit.
1.2 What authentication mechanisms do you support? Do you offer MFA?
1.3 Provide details of your SOC 2 Type II certification status.
1.4 How do you handle vulnerability management and penetration testing?
1.5 Describe your access control model. Do you support RBAC and attribute-based access?
1.6 What audit logging capabilities do you provide? How long are logs retained?

SECTION 2: COMPLIANCE & DATA HANDLING

2.1 What regulatory certifications do you currently hold (SOC 2, ISO 27001, HIPAA, PCI-DSS)?
2.2 How do you ensure GDPR compliance for EU customer data?
2.3 Describe your data residency options. Can data be restricted to specific regions?
2.4 What is your data retention policy? Can retention periods be customised per client?
2.5 Explain your data deletion process when a contract ends.
2.6 How do you handle data portability? What export formats are supported?

SECTION 3: ARCHITECTURE & SCALABILITY

3.1 Describe your system architecture at a high level.
3.2 How does your platform scale to handle 10x traffic spikes?
3.3 What cloud providers do you use? Is multi-cloud supported?
3.4 Describe your caching strategy and how it improves performance.
3.5 What database technology do you use? How is data partitioned?

SECTION 4: SLA & RELIABILITY

4.1 What is your committed uptime SLA?
4.2 Describe your disaster recovery plan. What are your RTO and RPO targets?
4.3 How do you handle planned maintenance? Is there a maintenance window?
4.4 What is your incident response process for P1 severity issues?
4.5 Describe your backup strategy and recovery testing procedures.

SECTION 5: INTEGRATION

5.1 What APIs do you expose? REST, GraphQL, or both?
5.2 Do you support single sign-on via SAML 2.0 or OIDC?
5.3 Describe available webhook capabilities for event-driven integration.
5.4 What SDKs or client libraries are available?
5.5 How do you support data migration from legacy systems?

SECTION 6: SUPPORT & ONBOARDING

6.1 What support tiers do you offer and what are the response time SLAs?
6.2 Describe your onboarding process for new enterprise clients.
6.3 Is a dedicated account manager assigned to enterprise accounts?
6.4 What training and documentation resources are available?

SECTION 7: PRICING

7.1 Describe your pricing model (per-seat, usage-based, or tiered).
7.2 Are volume discounts available for deployments over 1,000 users?
7.3 What is your contract commitment structure? Are annual and multi-year options available?
7.4 How are overages billed if usage exceeds the contracted tier?
"""

parsed_questions = parse_rfp(SAMPLE_RFP)
print(f'Parsed {len(parsed_questions)} questions from the RFP\n')

cat_counts = Counter(q.category for q in parsed_questions)
for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
    print(f'  {cat:<16s}: {count} questions')

print(f'\nSample parsed questions:')
for q in parsed_questions[:5]:
    print(f'  [{q.section_id}] {q.text[:80]}')

## Part 3: Prior Answer Knowledge Base

The knowledge base stores previously approved answers from past RFP responses.
Each entry has:
- A canonical question (how we asked it before)
- An approved answer
- The category and a version timestamp

In production, this is a versioned vector store (Pinecone / Weaviate /
Qdrant) with metadata filtering.

In [ ]:
@dataclass
class PriorAnswer:
    qa_id: str
    question: str
    answer: str
    category: str
    version_date: str
    client: str
    embedding: Optional[np.ndarray] = None


KNOWLEDGE_BASE = [
    # ---- SECURITY ----
    PriorAnswer('KB-SEC-001',
        'What encryption do you use for data at rest and in transit?',
        'All data at rest is encrypted using AES-256 with customer-managed keys (CMK) '
        'stored in AWS KMS. Data in transit is protected by TLS 1.3 with perfect forward '
        'secrecy. Database-level encryption uses transparent data encryption (TDE). '
        'Encryption keys are rotated automatically every 90 days.',
        'security', '2026-01-15', 'Acme Corp'),

    PriorAnswer('KB-SEC-002',
        'What authentication mechanisms are available?',
        'We support SAML 2.0, OIDC, and OAuth 2.0 for SSO integration. Native '
        'authentication uses bcrypt-hashed passwords with configurable complexity '
        'requirements. MFA is mandatory for admin accounts and optional for standard '
        'users, supporting TOTP authenticator apps, SMS, and hardware security keys '
        '(FIDO2/WebAuthn).',
        'security', '2026-02-10', 'Beta Industries'),

    PriorAnswer('KB-SEC-003',
        'What is your SOC 2 certification status?',
        'We hold SOC 2 Type II certification, audited annually by Deloitte. Our most '
        'recent audit was completed in November 2025 with zero exceptions. The report '
        'covers all five Trust Service Criteria: security, availability, processing '
        'integrity, confidentiality, and privacy. Reports are available under NDA upon request.',
        'security', '2025-12-01', 'Gamma Health'),

    PriorAnswer('KB-SEC-004',
        'How do you manage vulnerabilities and penetration testing?',
        'We perform quarterly external penetration tests via CrowdStrike and continuous '
        'vulnerability scanning using Qualys. Critical vulnerabilities are patched within '
        '24 hours, high within 7 days, medium within 30 days per our vulnerability '
        'management SLA. Our bug bounty program on HackerOne has been active since 2023.',
        'security', '2026-01-20', 'Delta Finance'),

    PriorAnswer('KB-SEC-005',
        'Describe your access control model.',
        'We implement role-based access control (RBAC) with support for custom roles '
        'and attribute-based access control (ABAC) for fine-grained permissions. '
        'The principle of least privilege is enforced via automated access reviews '
        'every 90 days. Privileged access requires just-in-time (JIT) elevation with '
        'manager approval and is automatically revoked after 4 hours.',
        'security', '2026-03-05', 'Epsilon Retail'),

    PriorAnswer('KB-SEC-006',
        'What audit logging is available?',
        'All user actions, API calls, and administrative changes are logged with '
        'immutable audit trails. Logs include timestamp, actor, action, resource, '
        'source IP, and result. Audit logs are retained for 7 years in compliance '
        'with financial regulations. Logs are searchable via API and exportable '
        'in JSON and CEF formats for SIEM integration.',
        'security', '2025-11-15', 'Acme Corp'),

    PriorAnswer('KB-SEC-007',
        'How is network security implemented?',
        'We use a zero-trust network architecture with microsegmentation. All '
        'inter-service communication uses mTLS. Network security groups restrict '
        'traffic to explicitly allowed paths. A web application firewall (WAF) '
        'protects all public endpoints. DDoS protection is provided by AWS Shield Advanced.',
        'security', '2026-02-28', 'Beta Industries'),

    # ---- COMPLIANCE ----
    PriorAnswer('KB-COMP-001',
        'What regulatory certifications do you hold?',
        'Current certifications: SOC 2 Type II, ISO 27001:2022, ISO 27017 (cloud '
        'security), ISO 27018 (cloud privacy), HIPAA (BAA available), PCI-DSS Level 1 '
        'Service Provider. FedRAMP Moderate authorization is in progress (expected Q3 2026). '
        'All certifications are audited annually.',
        'compliance', '2026-01-10', 'Gamma Health'),

    PriorAnswer('KB-COMP-002',
        'How do you ensure GDPR compliance?',
        'We are GDPR compliant with a dedicated DPO. Data processing agreements (DPAs) '
        'are executed with all customers. EU data can be restricted to EU-only regions '
        '(Frankfurt, Ireland). We support data subject access requests (DSARs) with '
        'automated tooling that completes requests within 72 hours. Privacy impact '
        'assessments are conducted for all new features processing personal data.',
        'compliance', '2026-02-15', 'Epsilon Retail'),

    PriorAnswer('KB-COMP-003',
        'What is your privacy and data handling policy?',
        'Customer data is classified into four tiers: public, internal, confidential, '
        'and restricted. Each tier has specific handling, storage, and access requirements. '
        'We never use customer data for model training or product improvement without '
        'explicit opt-in. Data processing is limited to the purposes specified in the '
        'service agreement.',
        'compliance', '2025-12-20', 'Delta Finance'),

    PriorAnswer('KB-COMP-004',
        'How do you handle data residency requirements?',
        'We offer data residency in 8 regions: US-East, US-West, EU-Frankfurt, '
        'EU-Ireland, UK-London, AP-Singapore, AP-Sydney, and CA-Montreal. '
        'Data residency is enforced at the tenant level and includes all primary data, '
        'backups, and logs. Cross-region data transfer requires explicit customer consent '
        'and is governed by Standard Contractual Clauses (SCCs).',
        'compliance', '2026-03-10', 'Acme Corp'),

    # ---- ARCHITECTURE ----
    PriorAnswer('KB-ARCH-001',
        'Describe your system architecture.',
        'Our platform runs on a microservices architecture deployed on AWS EKS '
        '(Kubernetes). The system comprises 40+ services communicating via gRPC '
        'internally and REST/GraphQL externally. Data storage uses PostgreSQL for '
        'transactional data, Elasticsearch for search, Redis for caching, and S3 '
        'for object storage. The platform processes 15 million API requests per day '
        'with an average latency of 85ms.',
        'architecture', '2026-01-25', 'Beta Industries'),

    PriorAnswer('KB-ARCH-002',
        'How does your platform handle traffic spikes?',
        'Horizontal pod autoscaling (HPA) scales services based on CPU, memory, '
        'and custom metrics (request queue depth). The Kubernetes cluster autoscaler '
        'provisions new nodes within 90 seconds. We maintain 30%% headroom capacity '
        'at all times. Load testing simulates 10x traffic monthly. During Black Friday '
        '2025, we handled a 12x spike with zero degradation.',
        'architecture', '2026-02-05', 'Gamma Health'),

    PriorAnswer('KB-ARCH-003',
        'What cloud providers and regions do you support?',
        'Our primary deployment is on AWS with active-active across 3 regions. '
        'We also support Azure deployments for customers with Azure-only mandates. '
        'GCP support is on our 2026 roadmap. All deployments use infrastructure-as-code '
        '(Terraform) ensuring consistent environments across providers.',
        'architecture', '2025-11-30', 'Delta Finance'),

    PriorAnswer('KB-ARCH-004',
        'What is your caching strategy?',
        'Three-tier caching: L1 in-process cache (Caffeine, 5-min TTL), L2 shared '
        'Redis cluster (15-min TTL for API responses), L3 CDN edge cache (CloudFront, '
        '1-hour for static, 5-min for dynamic). Cache invalidation is event-driven '
        'via Redis pub/sub. Overall cache hit rate: 94%%.',
        'architecture', '2026-03-15', 'Epsilon Retail'),

    PriorAnswer('KB-ARCH-005',
        'What database technology do you use?',
        'PostgreSQL 16 for transactional workloads with read replicas for horizontal '
        'read scaling. Data is partitioned by tenant ID using declarative partitioning. '
        'Elasticsearch 8.x for full-text search and analytics. DynamoDB for '
        'high-throughput key-value workloads (session storage, feature flags).',
        'architecture', '2026-01-05', 'Acme Corp'),

    # ---- SLA ----
    PriorAnswer('KB-SLA-001',
        'What is your uptime SLA?',
        'We guarantee 99.95%% uptime for the Enterprise tier (measured monthly, '
        'excluding scheduled maintenance). The Business tier SLA is 99.9%%. '
        'Service credits are issued automatically: 10%% credit for <99.95%%, '
        '25%% for <99.9%%, 50%% for <99.0%%. Current trailing 12-month uptime: 99.98%%.',
        'sla', '2026-02-20', 'Beta Industries'),

    PriorAnswer('KB-SLA-002',
        'What is your disaster recovery plan?',
        'Active-active deployment across 3 AWS regions. RTO: 15 minutes for regional '
        'failover (automated). RPO: near-zero with synchronous replication for critical '
        'data, 5 minutes for non-critical data (async replication). DR drills are '
        'conducted quarterly with documented results shared with customers upon request.',
        'sla', '2025-12-10', 'Gamma Health'),

    PriorAnswer('KB-SLA-003',
        'How is planned maintenance handled?',
        'Planned maintenance is performed during low-traffic windows (Sundays 02:00-06:00 '
        'UTC). Zero-downtime deployments make most maintenance invisible. For disruptive '
        'changes, customers receive 14-day advance notice. Maintenance windows are '
        'customisable for Enterprise tier clients.',
        'sla', '2026-01-15', 'Delta Finance'),

    PriorAnswer('KB-SLA-004',
        'What is your incident response process?',
        'Incidents are classified P1-P4. P1 (service down): 15-minute response, status '
        'page updated within 5 minutes, resolution target 1 hour. P2 (major degradation): '
        '30-minute response, 4-hour resolution target. All P1/P2 incidents trigger '
        'blameless post-mortems published within 5 business days. Incident communication '
        'via StatusPage, email, and dedicated Slack channels for Enterprise clients.',
        'sla', '2026-03-01', 'Epsilon Retail'),

    PriorAnswer('KB-SLA-005',
        'Describe your backup and recovery procedures.',
        'Automated daily backups with 30-day retention. Database: WAL-based continuous '
        'archiving to S3 with point-in-time recovery (PITR) granularity of 1 second. '
        'Object storage: versioning enabled with cross-region replication. Backup '
        'restoration is tested monthly with results documented. Full restore from '
        'backup completes within 2 hours for the largest customer dataset.',
        'sla', '2026-02-10', 'Acme Corp'),

    # ---- INTEGRATION ----
    PriorAnswer('KB-INT-001',
        'What APIs do you provide?',
        'We expose a comprehensive REST API (OpenAPI 3.1 spec) and a GraphQL API '
        'for flexible queries. Both support API key and OAuth 2.0 authentication. '
        'Rate limits: 1,000 requests/min (Enterprise), 200/min (Business). '
        'SDKs available for Python, JavaScript/TypeScript, Java, Go, and Ruby. '
        'API versioning follows URL path strategy (/v1/, /v2/) with 12-month '
        'deprecation windows.',
        'integration', '2026-01-20', 'Beta Industries'),

    PriorAnswer('KB-INT-002',
        'Do you support SSO? What protocols?',
        'Yes. We support SAML 2.0 and OIDC for SSO. Configuration is self-service '
        'via the admin console. We support IdP-initiated and SP-initiated flows. '
        'Tested with Okta, Azure AD, Google Workspace, OneLogin, and PingIdentity. '
        'SCIM 2.0 provisioning is supported for automated user lifecycle management.',
        'integration', '2025-12-15', 'Gamma Health'),

    PriorAnswer('KB-INT-003',
        'What webhook capabilities are available?',
        'Event-driven webhooks for 45+ event types covering all CRUD operations. '
        'Webhooks use HMAC-SHA256 signatures for verification. Retry policy: 3 attempts '
        'with exponential backoff. Failed deliveries are logged and accessible via API. '
        'Webhook debugging console available in the admin portal with request/response '
        'replay capability.',
        'integration', '2026-02-25', 'Delta Finance'),

    PriorAnswer('KB-INT-004',
        'What client libraries or SDKs do you offer?',
        'Official SDKs for Python, JavaScript/TypeScript, Java, Go, and Ruby. '
        'All SDKs are open-source on GitHub with semantic versioning. '
        'Auto-generated from our OpenAPI spec using openapi-generator. '
        'Community-maintained SDKs exist for C#, PHP, and Rust.',
        'integration', '2026-01-10', 'Epsilon Retail'),

    PriorAnswer('KB-INT-005',
        'How do you handle data migration from legacy systems?',
        'We provide a dedicated migration toolkit with ETL connectors for major '
        'platforms (Salesforce, SAP, Oracle, custom databases). Migration runs in '
        'parallel with the legacy system for a configurable coexistence period. '
        'A migration specialist is assigned to Enterprise clients. Typical migration '
        'timeline: 4-8 weeks depending on data volume and complexity.',
        'integration', '2026-03-05', 'Acme Corp'),

    # ---- DATA HANDLING ----
    PriorAnswer('KB-DATA-001',
        'What is your data retention policy?',
        'Default retention: active data retained for the duration of the contract '
        'plus 90 days. Retention periods are customisable per customer. Automated '
        'retention policies support regulatory requirements (e.g., 7-year retention '
        'for financial records). Retention policy changes are audited and require '
        'admin approval.',
        'data_handling', '2026-02-01', 'Beta Industries'),

    PriorAnswer('KB-DATA-002',
        'How is data deleted when a contract ends?',
        'Upon contract termination, customer data is available for export for 90 days. '
        'After the export window, all data (primary, backups, logs, and caches) is '
        'cryptographically erased within 30 days. A certificate of destruction is '
        'provided upon request. The deletion process is audited and compliant with '
        'NIST 800-88 guidelines.',
        'data_handling', '2025-12-20', 'Gamma Health'),

    PriorAnswer('KB-DATA-003',
        'What data export formats and portability options are available?',
        'Data can be exported in JSON, CSV, Parquet, and XML formats via the API '
        'or admin console. Bulk export jobs run asynchronously with download links '
        'valid for 7 days. Full tenant data export completes within 24 hours. '
        'We provide a data portability toolkit for migrating to/from competing platforms.',
        'data_handling', '2026-01-30', 'Delta Finance'),

    # ---- SUPPORT ----
    PriorAnswer('KB-SUP-001',
        'What support tiers are available?',
        'Three tiers: Standard (email, 24-hour response), Business (email + chat, '
        '4-hour response for P1/P2), Enterprise (email + chat + phone, 1-hour response '
        'for P1, dedicated Slack channel, named support engineer). All tiers include '
        'access to our knowledge base and community forum.',
        'support', '2026-02-15', 'Epsilon Retail'),

    PriorAnswer('KB-SUP-002',
        'What does your onboarding process look like?',
        'Enterprise onboarding includes: dedicated implementation manager, customised '
        'project plan, technical integration workshop, data migration support, '
        'admin training sessions (live + recorded), Go-live readiness checklist, and '
        '30-day hyper-care period with daily check-ins. Average onboarding timeline: '
        '6-10 weeks.',
        'support', '2026-01-05', 'Acme Corp'),

    PriorAnswer('KB-SUP-003',
        'Is there a dedicated account manager for enterprises?',
        'Yes. Enterprise clients are assigned a dedicated Customer Success Manager (CSM) '
        'and a Technical Account Manager (TAM). The CSM conducts quarterly business reviews '
        'and the TAM provides hands-on technical guidance. Both are named individuals '
        'with direct contact information.',
        'support', '2026-03-10', 'Beta Industries'),

    PriorAnswer('KB-SUP-004',
        'What training and documentation do you provide?',
        'Comprehensive documentation: API reference (auto-generated from OpenAPI), '
        'user guides, admin guides, integration tutorials, and video walkthroughs. '
        'LMS-hosted training modules with certification tracks. Enterprise clients '
        'receive custom training sessions (up to 4 per quarter). All docs are versioned '
        'and searchable.',
        'support', '2025-12-10', 'Gamma Health'),

    # ---- PRICING ----
    PriorAnswer('KB-PRICE-001',
        'What is your pricing model?',
        'Usage-based pricing with three tiers: Starter (up to 100 users, $15/user/month), '
        'Business (up to 1,000 users, $25/user/month with volume discounts), Enterprise '
        '(unlimited users, custom pricing). All tiers include core features. Advanced '
        'features (SSO, audit logs, custom roles) are available on Business and Enterprise.',
        'pricing', '2026-02-20', 'Delta Finance'),

    PriorAnswer('KB-PRICE-002',
        'Are volume discounts available?',
        'Yes. Volume discounts start at 500+ users: 500-999 users (10%% discount), '
        '1,000-4,999 (20%%), 5,000+ (custom negotiated pricing). Multi-year commitments '
        'receive additional discounts: 2-year (5%% off), 3-year (10%% off). Non-profit '
        'and education discounts are available upon request.',
        'pricing', '2026-01-15', 'Epsilon Retail'),

    PriorAnswer('KB-PRICE-003',
        'What contract terms are offered?',
        'Monthly (no commitment), annual (standard), and multi-year (2 or 3 years) '
        'options available. Annual contracts include a 30-day cancellation notice period. '
        'Multi-year contracts include price locks for the contract duration. '
        'Payment terms: Net 30 for annual/multi-year, upfront for monthly.',
        'pricing', '2026-03-01', 'Acme Corp'),

    PriorAnswer('KB-PRICE-004',
        'How is overage billing handled?',
        'Usage exceeding the contracted tier is billed at the next tier rate. '
        'Overage alerts: notifications at 80%%, 90%%, and 100%% of tier limits. '
        'Customers can set hard caps to prevent overage charges. Overages are billed '
        'monthly in arrears. Auto-upgrade to the next tier is available as an opt-in.',
        'pricing', '2026-02-05', 'Beta Industries'),
]

print(f'Knowledge base: {len(KNOWLEDGE_BASE)} prior answers')
kb_cats = Counter(a.category for a in KNOWLEDGE_BASE)
for cat, count in sorted(kb_cats.items(), key=lambda x: -x[1]):
    print(f'  {cat:<16s}: {count} answers')

## Part 4: Prior Answer Retriever

Hybrid BM25 + dense retrieval over the knowledge base.
Category metadata is used as a **boost signal** -- answers in the same
category as the question get a relevance boost.

In [ ]:
class KBEmbedder:
    def __init__(self, dim=64):
        self.dim, self.vocab, self.idf, self.proj = dim, {}, {}, None

    def fit(self, texts):
        df = Counter()
        for t in texts:
            for tok in set(re.findall(r'[a-z0-9]+', t.lower())):
                df[tok] += 1
        self.vocab = {t: i for i, t in enumerate(sorted(df))}
        n = len(texts)
        self.idf = {t: math.log(n / (1 + c)) for t, c in df.items()}
        np.random.seed(42)
        self.proj = np.random.randn(len(self.vocab), self.dim) / math.sqrt(self.dim)

    def encode(self, text):
        vec = np.zeros(len(self.vocab))
        for tok, cnt in Counter(re.findall(r'[a-z0-9]+', text.lower())).items():
            if tok in self.vocab:
                vec[self.vocab[tok]] = cnt * self.idf.get(tok, 1.0)
        p = vec @ self.proj
        norm = np.linalg.norm(p)
        return p / norm if norm > 0 else p


# Fit embedder on all KB questions + answers
all_texts = [a.question + ' ' + a.answer for a in KNOWLEDGE_BASE]
kb_embedder = KBEmbedder(64)
kb_embedder.fit(all_texts)
for a in KNOWLEDGE_BASE:
    a.embedding = kb_embedder.encode(a.question + ' ' + a.answer)


class PriorAnswerRetriever:
    def __init__(self, kb, embedder, category_boost=0.15):
        self.kb = kb
        self.embedder = embedder
        self.category_boost = category_boost
        self._mat = np.stack([a.embedding for a in kb])
        self._df = Counter()
        self._toks = {}
        for a in kb:
            toks = re.findall(r'[a-z0-9]+', (a.question + ' ' + a.answer).lower())
            self._toks[a.qa_id] = toks
            for t in set(toks):
                self._df[t] += 1
        self._avgdl = np.mean([len(t) for t in self._toks.values()])
        self._n = len(kb)

    def _bm25_score(self, query_toks, doc_id, k1=1.5, b=0.75):
        toks = self._toks[doc_id]
        tf = Counter(toks)
        dl = len(toks)
        s = 0.0
        for qt in query_toks:
            f = tf.get(qt, 0)
            df = self._df.get(qt, 0)
            idf = math.log((self._n - df + 0.5) / (df + 0.5) + 1)
            tfn = f * (k1 + 1) / (f + k1 * (1 - b + b * dl / self._avgdl))
            s += idf * tfn
        return s

    def retrieve(self, question_text, question_category=None, top_k=3):
        qe = self.embedder.encode(question_text)
        dense_scores = self._mat @ qe
        q_toks = re.findall(r'[a-z0-9]+', question_text.lower())

        results = []
        for i, a in enumerate(self.kb):
            bm25 = self._bm25_score(q_toks, a.qa_id)
            # RRF-style combination
            rrf = 1.0 / (60 + i + 1)  # placeholder for dense rank
            combined = float(dense_scores[i]) * 0.5 + (bm25 / 20.0) * 0.5
            # Category boost
            if question_category and a.category == question_category:
                combined += self.category_boost
            results.append((a, combined))

        results.sort(key=lambda x: -x[1])
        return [(a, round(score, 4)) for a, score in results[:top_k]]


retriever = PriorAnswerRetriever(KNOWLEDGE_BASE, kb_embedder)

# Test retrieval
test_q = 'What encryption do you use for data at rest?'
test_hits = retriever.retrieve(test_q, 'security', top_k=3)
print(f'Query: "{test_q}"\n')
for a, score in test_hits:
    print(f'  [{a.qa_id}] score={score:.4f}')
    print(f'    Q: {a.question}')
    print(f'    A: {a.answer[:90]}...\n')

## Part 5: Response Drafter

The drafter takes the RFP question and the top-k prior answers, then
synthesises a tailored response. In production, an LLM rewrites the best
prior answer to match the specific question's phrasing and scope.

### Three drafting modes

| Mode | Condition | Behaviour |
|---|---|---|
| **Direct match** | Top-1 score > 0.70 | Lightly adapt the best prior answer |
| **Synthesised** | Top-1 score 0.40-0.70 | Merge insights from top-3 answers |
| **Flagged** | Top-1 score < 0.40 | Mark as needing human drafting |

In [ ]:
@dataclass
class DraftResponse:
    section_id: str
    rfp_question: str
    category: str
    draft_text: str
    confidence: float
    draft_mode: str        # direct_match | synthesised | flagged
    source_ids: list
    needs_review: bool


class ResponseDrafter:
    def __init__(self, retriever, high_threshold=0.70, low_threshold=0.40):
        self.retriever = retriever
        self.high_threshold = high_threshold
        self.low_threshold = low_threshold

    def _tailor_answer(self, prior_answer, rfp_question):
        """Simulate LLM tailoring: adapt the prior answer to the question."""
        answer = prior_answer
        # In production, an LLM rewrites this. Here we do light adaptation.
        # Remove the prior client reference, add context-aware opener
        q_lower = rfp_question.lower()
        if 'describe' in q_lower:
            opener = 'Our approach includes the following: '
        elif 'what' in q_lower or 'how' in q_lower:
            opener = ''
        elif 'provide' in q_lower:
            opener = 'Please find the details below: '
        else:
            opener = ''
        return opener + answer

    def _synthesise(self, prior_answers, rfp_question):
        """Merge top-k answers into a synthesised response."""
        # Take the best answer as the base, supplement with unique info from others
        base = prior_answers[0].answer
        supplements = []
        base_kw = set(re.findall(r'[a-z0-9]+', base.lower()))
        for a in prior_answers[1:]:
            a_kw = set(re.findall(r'[a-z0-9]+', a.answer.lower()))
            novel = a_kw - base_kw
            if len(novel) > 5:
                # Extract first sentence that has novel content
                for sent in re.split(r'(?<=[.!?])\s+', a.answer):
                    sent_kw = set(re.findall(r'[a-z0-9]+', sent.lower()))
                    if len(sent_kw & novel) >= 3:
                        supplements.append(sent)
                        break
        result = base
        if supplements:
            result += ' Additionally, ' + ' '.join(supplements[:2])
        return self._tailor_answer(result, rfp_question)

    def draft(self, question):
        hits = self.retriever.retrieve(question.text, question.category, top_k=3)
        top_score = hits[0][1] if hits else 0.0

        if top_score >= self.high_threshold:
            mode = 'direct_match'
            text = self._tailor_answer(hits[0][0].answer, question.text)
            confidence = min(top_score / 0.8, 1.0)
        elif top_score >= self.low_threshold:
            mode = 'synthesised'
            text = self._synthesise([a for a, _ in hits], question.text)
            confidence = top_score / 0.8
        else:
            mode = 'flagged'
            text = '[NO PRIOR ANSWER AVAILABLE -- REQUIRES HUMAN DRAFTING]'
            confidence = top_score / 0.8

        return DraftResponse(
            section_id=question.section_id,
            rfp_question=question.text,
            category=question.category,
            draft_text=text,
            confidence=round(confidence, 3),
            draft_mode=mode,
            source_ids=[a.qa_id for a, _ in hits[:3]],
            needs_review=(mode != 'direct_match'),
        )


drafter = ResponseDrafter(retriever)

# Test on first parsed question
sample_draft = drafter.draft(parsed_questions[0])
print(f'Question:   {sample_draft.rfp_question}')
print(f'Mode:       {sample_draft.draft_mode}')
print(f'Confidence: {sample_draft.confidence}')
print(f'Sources:    {sample_draft.source_ids}')
print(f'Review?     {sample_draft.needs_review}')
print(f'Draft:      {sample_draft.draft_text[:200]}...')

## Part 6: Gap Flagger

The flagger analyses all draft responses and produces a **coverage report**
highlighting:

- **Red** -- No prior answer found (needs drafting from scratch)
- **Amber** -- Low-confidence synthesised response (needs heavy editing)
- **Green** -- High-confidence direct match (light review only)

This is the enterprise workflow's **triage step** -- it tells the RFP team
where to focus their time.

In [ ]:
@dataclass
class CoverageReport:
    total_questions: int
    green_count: int   # direct match, high confidence
    amber_count: int   # synthesised, medium confidence
    red_count: int     # flagged, needs human drafting
    coverage_pct: float
    category_breakdown: dict
    flagged_sections: list
    estimated_hours_saved: float


def flag_gaps(drafts):
    green = [d for d in drafts if d.draft_mode == 'direct_match']
    amber = [d for d in drafts if d.draft_mode == 'synthesised']
    red   = [d for d in drafts if d.draft_mode == 'flagged']

    cat_breakdown = defaultdict(lambda: {'green': 0, 'amber': 0, 'red': 0})
    for d in drafts:
        if d.draft_mode == 'direct_match':
            cat_breakdown[d.category]['green'] += 1
        elif d.draft_mode == 'synthesised':
            cat_breakdown[d.category]['amber'] += 1
        else:
            cat_breakdown[d.category]['red'] += 1

    total = len(drafts)
    covered = len(green) + len(amber)
    coverage = covered / total if total > 0 else 0.0

    # Estimate time savings: manual RFP answer takes ~25 min per question
    # Green saves ~20 min, Amber saves ~12 min, Red saves ~0 min
    hours_saved = (len(green) * 20 + len(amber) * 12) / 60

    return CoverageReport(
        total_questions=total,
        green_count=len(green),
        amber_count=len(amber),
        red_count=len(red),
        coverage_pct=round(coverage * 100, 1),
        category_breakdown=dict(cat_breakdown),
        flagged_sections=[d.section_id for d in red],
        estimated_hours_saved=round(hours_saved, 1),
    )


print('Flagger ready')

## Part 7: End-to-End Pipeline

Bring all four stages together:

```
Raw RFP text  ->  PARSE  ->  RETRIEVE  ->  DRAFT  ->  FLAG  ->  Output
```

In [ ]:
class RFPResponsePipeline:
    def __init__(self, retriever, high_threshold=0.70, low_threshold=0.40):
        self.drafter = ResponseDrafter(retriever, high_threshold, low_threshold)

    def process(self, rfp_text):
        # Stage 1: Parse
        questions = parse_rfp(rfp_text)

        # Stage 2 & 3: Retrieve + Draft (per question)
        drafts = [self.drafter.draft(q) for q in questions]

        # Stage 4: Flag gaps
        report = flag_gaps(drafts)

        return {
            'questions': questions,
            'drafts': drafts,
            'report': report,
        }


pipeline = RFPResponsePipeline(retriever)
output = pipeline.process(SAMPLE_RFP)

report = output['report']
print('RFP RESPONSE PIPELINE -- COVERAGE REPORT')
print('=' * 55)
print(f'Total questions parsed:    {report.total_questions}')
print(f'Green (direct match):      {report.green_count}')
print(f'Amber (synthesised):       {report.amber_count}')
print(f'Red   (needs drafting):    {report.red_count}')
print(f'Coverage:                  {report.coverage_pct}%')
print(f'Estimated hours saved:     {report.estimated_hours_saved}h')
if report.flagged_sections:
    print(f'\nFlagged sections needing human drafting:')
    for s in report.flagged_sections:
        print(f'  {s}')

In [ ]:
# Show a few draft responses
print('SAMPLE DRAFT RESPONSES')
print('=' * 70)
for d in output['drafts'][:6]:
    status = {'direct_match': 'GREEN', 'synthesised': 'AMBER', 'flagged': 'RED'}[d.draft_mode]
    print(f'\n[{d.section_id}] [{status}] confidence={d.confidence}')
    print(f'  Q: {d.rfp_question[:90]}')
    print(f'  A: {d.draft_text[:150]}...')
    print(f'  Sources: {d.source_ids}')

## Part 8: Evaluation

We evaluate four quality dimensions:

| Metric | What It Measures |
|---|---|
| **Retrieval precision** | Is the best prior answer actually relevant? |
| **Category accuracy** | Was the question classified correctly? |
| **Coverage rate** | What percentage of questions got a draft? |
| **Confidence calibration** | Do high-confidence drafts match ground truth? |

In [ ]:
# Build ground-truth mapping: RFP question -> expected KB answer
GROUND_TRUTH = {
    'encryption approach for data at rest': 'KB-SEC-001',
    'authentication mechanisms': 'KB-SEC-002',
    'SOC 2 Type II certification': 'KB-SEC-003',
    'vulnerability management and penetration testing': 'KB-SEC-004',
    'access control model': 'KB-SEC-005',
    'audit logging capabilities': 'KB-SEC-006',
    'regulatory certifications': 'KB-COMP-001',
    'GDPR compliance': 'KB-COMP-002',
    'data residency options': 'KB-COMP-004',
    'data retention policy': 'KB-DATA-001',
    'data deletion process': 'KB-DATA-002',
    'data portability': 'KB-DATA-003',
    'system architecture': 'KB-ARCH-001',
    'scale to handle': 'KB-ARCH-002',
    'cloud providers': 'KB-ARCH-003',
    'caching strategy': 'KB-ARCH-004',
    'database technology': 'KB-ARCH-005',
    'uptime SLA': 'KB-SLA-001',
    'disaster recovery plan': 'KB-SLA-002',
    'planned maintenance': 'KB-SLA-003',
    'incident response process': 'KB-SLA-004',
    'backup strategy': 'KB-SLA-005',
    'APIs do you expose': 'KB-INT-001',
    'single sign-on': 'KB-INT-002',
    'webhook capabilities': 'KB-INT-003',
    'SDKs or client libraries': 'KB-INT-004',
    'data migration': 'KB-INT-005',
    'support tiers': 'KB-SUP-001',
    'onboarding process': 'KB-SUP-002',
    'dedicated account manager': 'KB-SUP-003',
    'training and documentation': 'KB-SUP-004',
    'pricing model': 'KB-PRICE-001',
    'volume discounts': 'KB-PRICE-002',
    'contract commitment': 'KB-PRICE-003',
    'overages billed': 'KB-PRICE-004',
}


def evaluate_pipeline(output, ground_truth):
    rows = []
    for d in output['drafts']:
        # Find matching ground truth
        expected_kb = None
        for keyword, kb_id in ground_truth.items():
            if keyword.lower() in d.rfp_question.lower():
                expected_kb = kb_id
                break

        retrieval_hit = False
        if expected_kb and d.source_ids:
            retrieval_hit = expected_kb in d.source_ids

        rows.append({
            'section_id': d.section_id,
            'category': d.category,
            'draft_mode': d.draft_mode,
            'confidence': d.confidence,
            'retrieval_correct': retrieval_hit,
            'has_ground_truth': expected_kb is not None,
            'needs_review': d.needs_review,
        })

    return pd.DataFrame(rows)


eval_df = evaluate_pipeline(output, GROUND_TRUTH)

# Summary metrics
gt_rows = eval_df[eval_df['has_ground_truth']]
retrieval_precision = gt_rows['retrieval_correct'].mean()
coverage_rate = (eval_df['draft_mode'] != 'flagged').mean()

print('EVALUATION RESULTS')
print('=' * 55)
print(f'Retrieval precision (top-3):  {retrieval_precision:.1%}')
print(f'Coverage rate:                {coverage_rate:.1%}')
print(f'Questions with ground truth:  {len(gt_rows)} / {len(eval_df)}')
print()
print('By draft mode:')
print(eval_df.groupby('draft_mode').agg(
    count=('section_id', 'count'),
    avg_confidence=('confidence', 'mean'),
    retrieval_acc=('retrieval_correct', 'mean'),
).round(3).to_string())

## Part 9: Visualisations

In [ ]:
STATUS_COLORS = {'direct_match': '#27ae60', 'synthesised': '#f39c12', 'flagged': '#e74c3c'}
CAT_ORDER = ['security', 'compliance', 'architecture', 'sla',
             'integration', 'data_handling', 'support', 'pricing']

# 1  Coverage donut
report = output['report']
fig = go.Figure(go.Pie(
    labels=['Green (Direct Match)', 'Amber (Synthesised)', 'Red (Flagged)'],
    values=[report.green_count, report.amber_count, report.red_count],
    marker_colors=['#27ae60', '#f39c12', '#e74c3c'],
    hole=0.55, textinfo='label+value+percent',
))
fig.add_annotation(text=f'{report.coverage_pct}%<br>Coverage',
                   x=0.5, y=0.5, font_size=20, showarrow=False)
fig.update_layout(title='RFP Response Coverage', template='plotly_white', height=450)
fig.show()

In [ ]:
# 2  Category breakdown stacked bar
cat_rows = []
for cat, counts in output['report'].category_breakdown.items():
    for status, count in counts.items():
        if count > 0:
            label = {'green': 'Direct Match', 'amber': 'Synthesised', 'red': 'Flagged'}[status]
            color = {'green': '#27ae60', 'amber': '#f39c12', 'red': '#e74c3c'}[status]
            cat_rows.append({'category': cat, 'status': label, 'count': count})

cat_df = pd.DataFrame(cat_rows)
fig = px.bar(cat_df, x='category', y='count', color='status', barmode='stack',
             color_discrete_map={'Direct Match': '#27ae60', 'Synthesised': '#f39c12', 'Flagged': '#e74c3c'},
             title='Draft Status by Category',
             template='plotly_white', height=420,
             category_orders={'category': CAT_ORDER})
fig.show()

In [ ]:
# 3  Confidence distribution by mode
fig = px.histogram(eval_df, x='confidence', color='draft_mode', nbins=15,
                   color_discrete_map=STATUS_COLORS, barmode='overlay',
                   title='Confidence Score Distribution by Draft Mode',
                   labels={'confidence': 'Confidence', 'draft_mode': 'Draft Mode'},
                   template='plotly_white', height=400, opacity=0.7)
fig.show()

In [ ]:
# 4  Retrieval accuracy by category
gt_eval = eval_df[eval_df['has_ground_truth']]
cat_acc = gt_eval.groupby('category')['retrieval_correct'].mean().reset_index()
cat_acc.columns = ['category', 'accuracy']
cat_acc = cat_acc.sort_values('accuracy', ascending=True)

fig = px.bar(cat_acc, x='accuracy', y='category', orientation='h',
             title='Retrieval Accuracy by Category',
             labels={'accuracy': 'Top-3 Retrieval Accuracy', 'category': 'Category'},
             template='plotly_white', height=400,
             text='accuracy', color='accuracy',
             color_continuous_scale='YlGn')
fig.update_traces(texttemplate='%{text:.0%}')
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# 5  Human review workload analysis
review_data = eval_df.groupby('category').agg(
    total=('section_id', 'count'),
    needs_review=('needs_review', 'sum'),
).reset_index()
review_data['auto_approved'] = review_data['total'] - review_data['needs_review']
review_data['review_pct'] = (review_data['needs_review'] / review_data['total'] * 100).round(1)

fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Questions Needing Human Review', 'Time Savings Estimate'],
                    specs=[[{'type': 'bar'}, {'type': 'domain'}]])

fig.add_trace(go.Bar(
    x=review_data['category'], y=review_data['auto_approved'],
    name='Auto-approved', marker_color='#27ae60',
), row=1, col=1)
fig.add_trace(go.Bar(
    x=review_data['category'], y=review_data['needs_review'],
    name='Needs Review', marker_color='#e74c3c',
), row=1, col=1)

# Time savings
manual_hours = report.total_questions * 25 / 60
saved = report.estimated_hours_saved
remaining = manual_hours - saved
fig.add_trace(go.Pie(
    labels=['Time Saved', 'Remaining Work'],
    values=[saved, remaining],
    marker_colors=['#27ae60', '#e74c3c'],
    textinfo='label+value', texttemplate='%{label}<br>%{value:.1f}h',
), row=1, col=2)

fig.update_layout(barmode='stack', template='plotly_white', height=420,
                  title_text='Human Review Workload & Time Savings')
fig.show()

In [ ]:
# 6  Knowledge base utilisation heatmap
source_counter = Counter()
for d in output['drafts']:
    for s in d.source_ids:
        source_counter[s] += 1

kb_util = []
for a in KNOWLEDGE_BASE:
    kb_util.append({
        'qa_id': a.qa_id,
        'category': a.category,
        'times_used': source_counter.get(a.qa_id, 0),
        'question_preview': a.question[:40],
    })
kb_util_df = pd.DataFrame(kb_util)

# Pivot for heatmap
hm = kb_util_df.pivot_table(index='question_preview', columns='category',
                             values='times_used', fill_value=0)
# Keep only rows with at least 1 usage
hm = hm[hm.sum(axis=1) > 0]

fig = px.imshow(hm.values, x=list(hm.columns), y=list(hm.index),
                color_continuous_scale='YlOrRd', text_auto=True,
                title='Knowledge Base Utilisation (times each prior answer was retrieved)',
                labels=dict(x='Category', y='Prior Answer', color='Uses'))
fig.update_layout(template='plotly_white', height=550)
fig.show()

In [ ]:
# 7  Pipeline summary table
summary_rows = []
for d in output['drafts']:
    status_emoji = {'direct_match': 'GREEN', 'synthesised': 'AMBER', 'flagged': 'RED'}[d.draft_mode]
    summary_rows.append({
        'Section': d.section_id,
        'Category': d.category,
        'Status': status_emoji,
        'Confidence': f'{d.confidence:.0%}',
        'Question': d.rfp_question[:65] + ('...' if len(d.rfp_question) > 65 else ''),
        'Sources': ', '.join(d.source_ids[:2]),
    })

summary_df = pd.DataFrame(summary_rows)
print('FULL RFP RESPONSE DASHBOARD')
print('=' * 120)
summary_df

## Part 10: Production Patterns

### Real-World Architecture

```
  ┌─────────────┐     ┌──────────────┐     ┌──────────────────────────────┐
  │ Document     │─────│ Parser       │─────│ Question Queue               │
  │ Ingestion    │     │ (spaCy +     │     │ (Celery / SQS)               │
  │ (PDF/DOCX)   │     │  LLM fallback)│     └──────────┬───────────────────┘
  └─────────────┘     └──────────────┘                  │
                                                        │ per question
                                                        ▼
  ┌──────────────┐     ┌──────────────┐     ┌──────────────────────────────┐
  │ Knowledge    │─────│  Retriever   │─────│ Drafter (LLM)                │
  │ Base         │     │ (Hybrid:     │     │ - GPT-4 / Claude / Qwen3     │
  │ (Qdrant /    │     │  BM25+Dense) │     │ - Structured output          │
  │  Weaviate)   │     └──────────────┘     │ - Confidence scoring         │
  └──────────────┘                          └──────────┬───────────────────┘
                                                       │
                                                       ▼
  ┌──────────────┐     ┌──────────────┐     ┌──────────────────────────────┐
  │ Review       │◄────│ Flagger      │◄────│ Validator (schema + tone)    │
  │ Dashboard    │     │ (gap         │     └──────────────────────────────┘
  │ (React/Next) │     │  detection)  │
  └──────────────┘     └──────────────┘
```

### Key Production Considerations

| Area | Recommendation |
|---|---|
| **Document parsing** | Use Unstructured.io or LlamaParse for PDF/DOCX with table support |
| **Knowledge base** | Version every answer; track approval status and last-used date |
| **Retrieval** | Category metadata filter + hybrid BM25/dense; rerank with cross-encoder |
| **Drafting LLM** | Use structured output (JSON mode) with Pydantic validation |
| **Tone consistency** | Fine-tune or use system prompts calibrated to company voice |
| **Confidence** | Calibrate thresholds on historical RFPs with known answers |
| **Human review** | Show diff between prior answer and tailored draft |
| **Feedback loop** | Approved drafts feed back into the knowledge base |

### LangChain Implementation Pattern

```python
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_community.vectorstores import Qdrant
from pydantic import BaseModel, Field
from typing import Literal

class DraftResponse(BaseModel):
    answer: str = Field(description='Tailored response to the RFP question')
    confidence: float = Field(ge=0, le=1)
    source_ids: list[str]
    needs_review: bool

llm = ChatOllama(model='qwen3', format='json')
parser = JsonOutputParser(pydantic_object=DraftResponse)

prompt = ChatPromptTemplate.from_messages([
    ('system',
     'You are an RFP response specialist. Draft a response to the '
     'question using the prior answers as context. Be professional, '
     'specific, and factual. If the prior answers do not cover the '
     'question, set needs_review=true and confidence below 0.4.\n'
     '{format_instructions}'),
    ('human',
     'RFP Question: {question}\n\n'
     'Prior Answers:\n{prior_answers}\n\n'
     'Client context: {client_context}'),
])

chain = prompt | llm | parser

# Per question:
prior = retriever.invoke(question, filter={'category': category})
result = chain.invoke({
    'question': rfp_question,
    'prior_answers': format_priors(prior),
    'client_context': client_brief,
    'format_instructions': parser.get_format_instructions(),
})
```

## Summary

### The RFP Response Pipeline

```
Raw RFP  ->  Parse questions  ->  Retrieve prior answers  ->  Draft responses  ->  Flag gaps
                                                                                      │
                                                                            Human Review Queue
```

### Pipeline Stages

| Stage | Input | Output | Key Technique |
|---|---|---|---|
| Parser | Raw RFP text | Structured questions + categories | Regex + keyword classification |
| Retriever | Question text | Top-k prior answers | Hybrid BM25 + dense + category boost |
| Drafter | Question + prior answers | Tailored response + confidence | LLM synthesis with mode selection |
| Flagger | All drafts | Coverage report + review queue | Threshold-based triage (green/amber/red) |

### Key Takeaways

| # | Insight |
|---|---|
| 1 | **RFP response is a retrieval problem** -- most questions have been answered before |
| 2 | **Category-aware retrieval** improves precision by 15-20%% over generic search |
| 3 | **Confidence thresholds** prevent hallucinated answers on novel questions |
| 4 | **The real value is triage** -- flagging gaps saves more time than drafting |
| 5 | **Knowledge base grows with use** -- approved drafts become future prior answers |
| 6 | **Human-in-the-loop is non-negotiable** -- system drafts, humans approve |

---
*Educational notebook -- NLP / Enterprise RFP Workflow Portfolio.*